In [ ]:
import os
import numpy as np
import json
import pydicom
import pyvista as pv
import matplotlib.pyplot as plt
# from skimage import exposure, filters
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import pyacvd
# from pyvistaqt import BackgroundPlotter
# from qtpy import QtWidgets
# from pathlib import Path
import suport_mri as ut
from scipy.interpolate import Rbf
from scipy.spatial import cKDTree
from inlet_boundary import rotate_vectors_to_source, write_inlet_velocity_file

In [ ]:
# Paper algorithms use the public config interface. All clinical values and paths
# come from git-ignored config/local.py; copy config/local_example.py for its schema.
from config import load_case

CASE = os.environ.get("AORTA_CASE")
if not CASE:
    raise RuntimeError("Set AORTA_CASE to a label defined in private config/local.py")
cfg = load_case(CASE)

estudo  = cfg["dataset_id"]
mean_hr = cfg["heart_rate_bpm"]

# Rigid transform aligning the cCTA geometry to the MRI volume, set visually from
# anatomical landmarks.
T_Tra = np.array(cfg["mri_translation"], dtype=float)
rotation_angles = np.radians(cfg["mri_rotation_deg"])

directory  = cfg["mesh_surfaces_dir"]
direc_main = os.path.join(cfg["sim_dir"], "mesh")

inlet_mesh = pv.read(os.path.join(directory, "inlet.vtp"))
out_mesh   = pv.read(os.path.join(directory, "out.vtp"))

# Load centerline
centerline = pv.read(cfg["centerline"])
centerline_points = centerline.points

# NOTE: previously these three paths hardcoded the case-C simulation folder while
# `estudo` was a variable, so they were correct only for the active case. They now
# derive from the selected private configuration.
ct_read      = os.path.join(cfg["imaging_dir"], "images") + os.sep
mri_read     = os.path.join(cfg["imaging_dir"], "mri") + os.sep
segment_read = cfg["deformed_mesh_dir"] + os.sep
print(segment_read)

infolder_ct_all = os.listdir(ct_read)
infolder_ct_all.sort(key=lambda f: int(''.join(filter(str.isdigit, f))))
infolder_ct = []
for folder in infolder_ct_all:
    if str.isnumeric(folder[0]):
        infolder_ct.append(folder)

infolder_mri = os.listdir(mri_read)
infolder_mri.sort(key=lambda f: int(''.join(filter(str.isdigit, f))))


In [ ]:
# # CT haH
# # Read a CT DICOM file and print the Heart Rate (HR) if available

# ct_dicom_path = os.path.join(ct_direct)
# ds_ct = pydicom.dcmread(ct_dicom_path)

# hr = ds_ct.get((0x0045, 0x1031), None)  # Tag for Heart Rate
# if hr is not None:
#     print(f"Heart Rate (HR) from CT DICOM: {hr} bpm")
# else:
#     print("Heart Rate (HR) tag not found in CT DICOM metadata.")

# # print(ds_ct)

# MRI haH
# Example usage
dicom_file = cfg["mri_venc_reference"]
# ds_mri = pydicom.dcmread(dicom_file)

# hr = ds_mri.get((0x0018, 0x1088), None) # Tag for Heart Rate

# if hr is not None:
#     print(f"Heart Rate (HR) from MRI DICOM: {hr} bpm")
# else:
#     print("Heart Rate (HR) tag not found in MRI DICOM metadata.")

In [ ]:
file_path = cfg["cuts_posit"]

# Check if the file exists
if not os.path.isfile(file_path):
    raise FileNotFoundError(f"The file '{file_path}' does not exist.")

data = []
with open(file_path, "r") as f:
    # Read the header
    header = f.readline().strip().split("\t")
    # Read the data rows
    for line in f:
        row = line.strip().split("	")
        first=row[1][1:-1].split(' ')
        second=row[2][1:-1].split(' ')
        
        def remotion(array):
            arrayR=[]
            for i in array:
                if not i=='':
                    arrayR.append(float(i))
            return arrayR
        
        first=remotion(first)
        second=remotion(second) 

        data.append([first,second])


plane_info_0=np.array(data[0])
print(plane_info_0)

# CT

In [ ]:
organized_files={}  
ct_infolder_total=os.listdir(segment_read)

for file_name in ct_infolder_total:
            if file_name.endswith(".vtp"):
                first_digit = int(file_name.split('_')[1].split('.')[0])  # Extract the first digit
                organized_files.setdefault(first_digit, []).append(file_name)


load_name_mesh,load_name_mesh_load=[],[] 

for digit in sorted(organized_files.keys(), key=lambda x: f"{x:03}"):
        load_name_mesh.append(organized_files[digit][0])
        load_name_mesh_load.append(segment_read + organized_files[digit][0])
    # print(sorted(organized_files.keys()))
meshes = []
for file in load_name_mesh_load:
            print(file)
            mesh_r=pv.read(file)
            # mesh_r=mesh_r.warp_by_vector("Displacement").clean().fill_holes(50)
            mesh_r=mesh_r.clean().fill_holes(50)
            meshes.append(mesh_r)

In [ ]:
print("Total meshes:", len(meshes))
print("Total meshes size:", len(meshes[0].points))

# MRI

In [ ]:
ds = pydicom.dcmread(dicom_file)

# Parâmetros extraídos do metadado
venc = ds[0x0019, 0x10E2].value # (0019,10E2) 
print(f"VENC (Velocity Encoding): {venc} cm/s")
v_scale = ds[0x0019, 0x1084].value
print(f"Velocity Scale: {v_scale} cm/s")

# Extração direta dos metadados de armazenamento
bits_stored = ds.BitsStored      
pixel_repr = ds.PixelRepresentation 
pixel_data = ds.pixel_array
actual_max = np.max(pixel_data)
actual_min = np.min(pixel_data)
v_measured_max = actual_max * v_scale
print(f"--- Dados de Pixel Detectados ---")
print(f"Bits Stored: {bits_stored}")
print(f"Pixel Representation: {'Signed' if pixel_repr == 1 else 'Unsigned'}")
print(f"Valor Máximo de Pixel na Imagem: {actual_max}")
print(f"Valor Mínimo de Pixel na Imagem: {actual_min}")

# Teste: O valor máximo de pixel multiplicado pela escala ultrapassa o VENC?
v_measured_max = actual_max * v_scale

print(f"\n--- Verificação de Escala ---")
print(f"Velocidade Máxima Medida (Pixel Max * Scale): {v_measured_max:.2f}")
print(f"VENC (Limite de Nyquist): {venc:.2f}")

if v_measured_max > venc * 1.1: # Tolerância de 10%
    print("Resultado: A escala provavelmente resulta em mm/s (ou o pixel não é normalizado).")
else:
    print("Resultado: A escala é consistente com cm/s.")

In [ ]:
data, meta = ut.read_acquisition(mri_read)
arrayData_orig = ut.seriesData_to_arrayData(data, meta)# shape: (Y, X, Z, T)
arrayData = arrayData_orig.copy()
meansArr = [np.mean(x) for x in arrayData]
dist0 = []
dist1 = []
dist2 = []
dist3 = []
for i in [1, 2, 3]:
    dist0.append(np.abs(meansArr[0] - meansArr[i]))
for i in [0, 2, 3]:
    dist1.append(np.abs(meansArr[1] - meansArr[i]))
for i in [0, 1, 3]:
    dist2.append(np.abs(meansArr[2] - meansArr[i]))
for i in [0, 1, 2]:
    dist3.append(np.abs(meansArr[3] - meansArr[i]))

allmean = [np.mean(d) for d in [dist0, dist1, dist2, dist3]]
magId = np.argmax(allmean)
magoring = arrayData[magId]
arrayData.pop(magId)
velOrig = np.zeros((meta['num_rows'], meta['num_cols'], meta['num_slices'], meta['num_frames'], 3))

for i in range(3):
    velOrig[:, :, :, :, i] = arrayData[i][:]

print('Adjusting units and scale.')
velOrig *= v_scale #to mm/s

In [ ]:
#  filtros e improvement 
magTemp = magoring.copy()
velTemp = velOrig.copy()

# Extract first timestep 3D volume
image_3d = magTemp[:, :, :, 5]

magTemp = np.swapaxes(magTemp, 0, 2)
velTemp = np.swapaxes(velTemp, 0, 2)


velTemp[:, :, :, :, 2] *= -1
# velTemp.shape

spacing_mm = np.array(meta["spacing"]) * 1000  # meters to mm
origin_mm = np.array(meta["origin"]) * 1000

In [ ]:
# ROTATIONS 
velTemp = np.rot90(velTemp, k=1, axes=(0, 2)) ### RODA y
velTemp = np.rot90(velTemp, k=-1, axes=(0, 1))  ### RODA z

magTemp = np.rot90(magTemp, k=1, axes=(0, 2)) ### RODA y
magTemp = np.rot90(magTemp, k=-1, axes=(0, 1))  ### RODA z

# plot CT e MRI original

In [ ]:
time_step = 5
mag_at_t5 = magTemp[:, :, :, time_step]

# Velocity components at time step 5
vel_x = velTemp[:, :, :, time_step, 0]
vel_y = velTemp[:, :, :, time_step, 1]
vel_z = velTemp[:, :, :, time_step, 2]

# Create PyVista ImageData
mri_grid = pv.ImageData()

mri_grid.dimensions = mag_at_t5.shape
mri_grid.spacing = spacing_mm
mri_grid.origin = origin_mm


# Add velocity as vector field
velocity_vectors = np.column_stack([
    vel_x.flatten(order='F'),
    vel_y.flatten(order='F'),
    vel_z.flatten(order='F')
])

mri_grid["Velocity"] = velocity_vectors
mri_grid["Magnitude"] = mag_at_t5.flatten(order='F')
# Save to VTK file
output_path = f"mri_timestep_{time_step}.vtk"
mri_grid.save(output_path)
print(f"Saved MRI volume to {output_path}")

In [ ]:
# T_Tra = np.array([162.801,175.818,227.087], dtype=float) 

# Build rotation matrix (Rz * Ry * Rx)
cx, cy, cz = np.cos(rotation_angles)
sx, sy, sz = np.sin(rotation_angles)

Rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]])
Ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]])
Rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]])

R_Tra = Rz @ Ry @ Rx

print("Rotation matrix R_Tra:")
print(R_Tra)
print(f"\nTranslation vector T_Tra: {T_Tra}")

# Define rotation center (use origin for consistency)
rotation_center = np.array([0.0, 0.0, 0.0])

def transform_point(pt, R=R_Tra, T=T_Tra, center=rotation_center):
    """Apply rotation around center, then translation to a single point."""
    pt = np.asarray(pt)
    pt_centered = pt - center
    pt_rotated = R @ pt_centered + center
    return pt_rotated + T

def apply_transform(mesh, R=R_Tra, T=T_Tra, center=rotation_center):
    """Apply rotation around center, then translation to a mesh."""
    transformed = mesh.copy()
    pts = transformed.points.copy()
    
    # Rotate around center: R @ (p - c) + c
    pts_centered = pts - center
    pts_rotated = (R @ pts_centered.T).T + center
    
    # Then translate
    pts_final = pts_rotated + T
    
    transformed.points = pts_final
    return transformed


In [ ]:
# Compute velocity magnitude for visualization
vel_mag_3d = np.sqrt(
    vel_x[:, :, :]**2 +
    vel_y[:, :, :]**2 +
    vel_z[:, :, :]**2
)

# Create velocity grid
vel_grid = pv.ImageData()
vel_grid.dimensions = vel_mag_3d.shape
vel_grid.spacing = [spacing_mm[0], spacing_mm[1], spacing_mm[2]]
vel_grid.origin = origin_mm
vel_grid["VelocityMag"] = vel_mag_3d.flatten(order="F")

# --- plane translation ---
# plane_info_0[0] = center (3-vector)
# plane_info_0[1] = normal (3-vector)

C = np.array(plane_info_0[0], dtype=float)
N = np.array(plane_info_0[1], dtype=float)
N = N / np.linalg.norm(N)

# Transform plane center and normal
C_new = transform_point(C)
N_new = R_Tra @ N  # Rotate normal vector (no translation)

# --- slice ---
slice_plane = mri_grid.slice(origin=C_new, normal=N_new)

# --- plot ---
mesh_T = apply_transform(meshes[0])

# p = pv.Plotter(notebook=True)
# p.add_mesh(apply_transform(meshes[0].warp_by_vector("Displacement")), color='lightgrey', opacity=0.2)
# p.add_mesh(slice_plane, scalars="Enhanced")
# # p.add_mesh(apply_transform(inlet_mesh), color='yellow', opacity=0.5,show_edges=True)
# p.add_volume(vel_grid, scalars="VelocityMag",opacity=[0, 0.5, 0.8, 1], cmap="jet", clim=[2, 20])
# p.show()

# Inlet flow computation from MRI velocity field



This section interpolates the 3D MRI velocity field onto the inlet surface for every time step and computes the volumetric flow rate through the inlet.

In [ ]:
def build_velocity_grid(velmasked, t, spacing_mm, origin_mm):
    """Build a PyVista ImageData grid with velocity vectors at time step t."""
    vel_x_t = velmasked[:, :, :, t, 0]
    vel_y_t = velmasked[:, :, :, t, 1]
    vel_z_t = velmasked[:, :, :, t, 2]

    vel_grid_t = pv.ImageData()
    vel_grid_t.dimensions = vel_x_t.shape
    vel_grid_t.spacing = spacing_mm
    vel_grid_t.origin = origin_mm

    velocity_vectors_t = np.column_stack([
        vel_x_t.flatten(order="F"),
        vel_y_t.flatten(order="F"),
        vel_z_t.flatten(order="F"),
    ])
    vel_grid_t["Velocity"] = velocity_vectors_t
    return vel_grid_t

def create_circle_on_plane(center, normal, radius, n_points=50):
    """Create a circle mesh lying on a plane defined by center and normal."""
    # Normalize the normal vector
    normal = normal / np.linalg.norm(normal)
    
    # Find two perpendicular vectors in the plane
    # Choose an arbitrary vector not parallel to normal
    if abs(normal[0]) < 0.9:
        arbitrary = np.array([1, 0, 0])
    else:
        arbitrary = np.array([0, 1, 0])
    
    # First perpendicular vector (in plane)
    v1 = np.cross(normal, arbitrary)
    v1 = v1 / np.linalg.norm(v1)
    
    # Second perpendicular vector (in plane)
    v2 = np.cross(normal, v1)
    v2 = v2 / np.linalg.norm(v2)
    
    # Generate circle points
    theta = np.linspace(0, 2*np.pi, n_points, endpoint=False)
    circle_points = np.zeros((n_points, 3))
    for i, t in enumerate(theta):
        circle_points[i] = center + radius * (np.cos(t) * v1 + np.sin(t) * v2)
    
    # Create a closed polyline
    lines = np.zeros((n_points + 1,), dtype=int)
    lines[0] = n_points
    lines[1:] = np.arange(n_points)
    
    circle = pv.PolyData(circle_points, lines=np.hstack([[n_points] + list(range(n_points)) + [0]]))
    return circle

def create_outlet_from_centerline(mesh, centerline_pts, point_idx, radius=None, name="outlet"):

    # Get origin and compute normal direction
    origin = centerline_pts[point_idx]
    next_pt = centerline_pts[point_idx + 1]
    normal = next_pt - origin
    normal = normal / np.linalg.norm(normal)  # normalize
    
    # Cut the mesh with a plane - this produces LINE cells, not polygons
    sliced = mesh.slice(normal=normal, origin=origin)
    
    if sliced.n_points == 0:
        print(f"{name}: WARNING - no points after slicing!")
        return sliced
    
    # Filter points within radius of origin (creates disk instead of infinite plane)
    distances = np.linalg.norm(sliced.points - origin, axis=1)
    
    # Auto-estimate radius if not provided
    if radius is None:
        connected = sliced.connectivity(extraction_mode='all')
        if 'RegionId' in connected.point_data:
            region_ids = np.unique(connected['RegionId'])
            min_dist = np.inf
            best_region = 0
            for rid in region_ids:
                mask = connected['RegionId'] == rid
                region_center = np.mean(connected.points[mask], axis=0)
                dist = np.linalg.norm(region_center - origin)
                if dist < min_dist:
                    min_dist = dist
                    best_region = rid
            best_mask = connected['RegionId'] == best_region
            best_distances = distances[best_mask]
            radius = np.max(best_distances) * 1.1
            print(f"{name}: auto-estimated radius = {radius:.2f} mm")
    
    # Extract points within radius
    within_radius = distances <= radius
    pts_within = sliced.points[within_radius]
    
    if len(pts_within) == 0:
        print(f"{name}: WARNING - no points within radius {radius}!")
        return pv.PolyData()
    
    # Create a filled surface using delaunay_2d triangulation
    # This converts the ring of points into a filled disk
    cloud = pv.PolyData(pts_within)
    result = cloud.delaunay_2d()
    
    # Clean up the result
    result = result.clean()
    
    area = result.compute_cell_sizes(length=False, area=True, volume=False)["Area"].sum()
    
    print(f"{name}: origin={origin.round(2)}, normal={normal.round(3)}, "
          f"radius={radius:.1f}mm, n_points={result.n_points}, n_cells={result.n_cells}, area={area:.1f}mm²")
    
    return result

# Create circles showing the radius boundaries
def get_normal_at_idx(centerline_pts, idx):
    origin = centerline_pts[idx]
    next_pt = centerline_pts[idx + 1]
    normal = next_pt - origin
    return normal / np.linalg.norm(normal)


In [ ]:
# Geometry-dependent centreline selections come from the private case config.
# Point indices depend on centreline sampling and therefore have no universal
# public default.
points_defined=True
outlet_sections = cfg["mri_outlet_sections"]
out1_idx = outlet_sections["out1"]["centerline_index"]
out1_radius = outlet_sections["out1"]["radius_mm"]
out2_idx = outlet_sections["out2"]["centerline_index"]
out2_radius = outlet_sections["out2"]["radius_mm"]
out3_idx = outlet_sections["out3"]["centerline_index"]
out3_radius = outlet_sections["out3"]["radius_mm"]
for name, index in (("out1", out1_idx), ("out2", out2_idx), ("out3", out3_idx)):
    if index >= len(centerline_points):
        raise IndexError(f"{name} centreline index {index} exceeds {len(centerline_points)} points")

# Create outlet meshes from centerline cuts with radius limits
out1_mesh = create_outlet_from_centerline(meshes[0], centerline_points, out1_idx, radius=out1_radius, name="out1 (BCT)")
out2_mesh = create_outlet_from_centerline(meshes[0], centerline_points, out2_idx, radius=out2_radius, name="out2 (LCCA)")
out3_mesh = create_outlet_from_centerline(meshes[0], centerline_points, out3_idx, radius=out3_radius, name="out3 (LSA)")


circle1 = create_circle_on_plane(centerline_points[out1_idx], get_normal_at_idx(centerline_points, out1_idx), out1_radius)
circle2 = create_circle_on_plane(centerline_points[out2_idx], get_normal_at_idx(centerline_points, out2_idx), out2_radius)
circle3 = create_circle_on_plane(centerline_points[out3_idx], get_normal_at_idx(centerline_points, out3_idx), out3_radius)

# Visualization
if points_defined:
    p = pv.Plotter(notebook=True)
else:
    p = pv.Plotter(notebook=False)
p.add_mesh(apply_transform(inlet_mesh), color="cyan", opacity=0.7, label="inlet")
p.add_mesh(apply_transform(out_mesh), color="magenta", opacity=0.7, label="outlet")
p.add_mesh(mri_grid.outline(), color='blue', line_width=2, label='MRI bounds')

# Add outlet meshes
if points_defined:
    p.add_mesh(apply_transform(out1_mesh), color="red", opacity=0.7, label="out1 (BCT)")
    p.add_mesh(apply_transform(out2_mesh), color="green", opacity=0.7, label="out2 (LCCA)")
    p.add_mesh(apply_transform(out3_mesh), color="yellow", opacity=0.7, label="out3 (LSA)")

# Add radius circles
p.add_mesh(apply_transform(circle1), color="red", line_width=3, render_lines_as_tubes=True)
p.add_mesh(apply_transform(circle2), color="green", line_width=3, render_lines_as_tubes=True)
p.add_mesh(apply_transform(circle3), color="yellow", line_width=3, render_lines_as_tubes=True)

# Add center points
p.add_points(np.array([transform_point(centerline_points[out1_idx])]), color="red", point_size=15, render_points_as_spheres=True)
p.add_points(np.array([transform_point(centerline_points[out2_idx])]), color="green", point_size=15, render_points_as_spheres=True)
p.add_points(np.array([transform_point(centerline_points[out3_idx])]), color="yellow", point_size=15, render_points_as_spheres=True)

p.add_mesh(apply_transform(centerline), color='white', line_width=2)
p.add_mesh(apply_transform(meshes[0]), color='lightgrey', opacity=0.2)

# Add index labels every 500 points along centerline
label_interval = 50
for idx in range(0, len(centerline_points), label_interval):
    point = transform_point(centerline_points[idx])
    p.add_point_labels(
        [point], 
        [f"{idx}"], 
        font_size=10, 
        point_color='cyan',
        point_size=8,
        render_points_as_spheres=True,
        text_color='white',
        shape_opacity=0.5,
        always_visible=True
    )

p.add_legend()
p.show()

RBF

In [ ]:
def rbf_calculation_def(mesh_t, slice_mesh):

    # Build KD-tree from mesh_t points
    tree = cKDTree(mesh_t.points)

    # Query ball around each slice_mesh point
    all_nearby_idx = set()
    for pt in slice_mesh.points:
        nearby = tree.query_ball_point(pt, r=1.0)  # 1mm radius
        all_nearby_idx.update(nearby)

    idx = np.array(list(all_nearby_idx))

    # Use these points for RBF
    src_pts = mesh_t.points[idx]
    src_disp = mesh_t.point_data["Displacement"][idx]
    
    x_src, y_src, z_src = src_pts.T
    dx, dy, dz = src_disp.T

    # Build RBF interpolators
    rbf_x = Rbf(x_src, y_src, z_src, dx, function='linear')
    rbf_y = Rbf(x_src, y_src, z_src, dy, function='linear')
    rbf_z = Rbf(x_src, y_src, z_src, dz, function='linear')

    # Interpolate displacement at slice points
    slice_pts = slice_mesh.points
    disp_x = rbf_x(slice_pts[:,0], slice_pts[:,1], slice_pts[:,2])
    disp_y = rbf_y(slice_pts[:,0], slice_pts[:,1], slice_pts[:,2])
    disp_z = rbf_z(slice_pts[:,0], slice_pts[:,1], slice_pts[:,2])
    
    # Create deformed mesh
    deformed = slice_mesh.copy()
    deformed.points = slice_pts + np.column_stack([disp_x, disp_y, disp_z])
    
    return deformed


# Precompute RBF interpolators for each mesh timestep
print("Building RBF interpolators for each timestep...")
rbf_interpolators_per_timestep = []


out1_meshes_deformed = [out1_mesh]
out2_meshes_deformed = [out2_mesh]
out3_meshes_deformed = [out3_mesh]
out_meshes_deformed = [out_mesh]
inlet_meshes_deformed = [inlet_mesh]

for t in tqdm(range(len(meshes)), desc="Building RBF interpolators"):
    if t==0:
        continue
    out1_meshes_deformed.append(rbf_calculation_def(meshes[t], out1_mesh))
    out2_meshes_deformed.append(rbf_calculation_def(meshes[t], out2_mesh))
    out3_meshes_deformed.append(rbf_calculation_def(meshes[t], out3_mesh))
    out_meshes_deformed.append(rbf_calculation_def(meshes[t], out_mesh))
    inlet_meshes_deformed.append(rbf_calculation_def(meshes[t], inlet_mesh))

p=pv.Plotter(notebook=True)
p.add_mesh(apply_transform(meshes[0]), color='blue', opacity=0.5)
for i in range(len(meshes)):
    if i==10:
        p.add_mesh(apply_transform(meshes[i].warp_by_vector("Displacement")), color='lightgrey', opacity=0.2)
        p.add_mesh(apply_transform(out1_meshes_deformed[i]), color="red", opacity=0.7)
        p.add_mesh(apply_transform(out2_meshes_deformed[i]), color="green", opacity=0.7)
        p.add_mesh(apply_transform(out3_meshes_deformed[i]), color="gray", opacity=0.7)
        p.add_mesh(apply_transform(out_meshes_deformed[i]), color="blue", opacity=0.7)
        p.add_mesh(apply_transform(inlet_meshes_deformed[i]), color="yellow", opacity=0.7)
p.show()

Interpolation

In [ ]:
# Compute flow rates using time-varying deformed surfaces
n_frames = velTemp.shape[3]  # Number of MRI timesteps

deformed_flow_t = {
    "inlet": []*20,
    "out": []*20,
    "out1": []*20,
    "out2": []*20,
    "out3": []*20,
}
# Map surface names to their deformed mesh lists
deformed_meshes = {
    "inlet": inlet_meshes_deformed,
    "out": out_meshes_deformed,
    "out1": out1_meshes_deformed,
    "out2": out2_meshes_deformed,
    "out3": out3_meshes_deformed,
}



# Match MRI timesteps to CT mesh timesteps
n_ct_meshes = len(meshes)

print(f"Computing flow rates for {n_frames} MRI timesteps using {n_ct_meshes} CT mesh timesteps...")

for t in tqdm(range(n_frames), desc="Computing flow rates"):
    # Build velocity grid for this MRI timestep
    vel_grid_t = build_velocity_grid(velTemp, t, spacing_mm, origin_mm)
    
    # Map MRI timestep to CT mesh index (interpolate if different number of frames)
    ct_idx = min(int(t * n_ct_meshes / n_frames), n_ct_meshes - 1)
    
    # Build surfaces dict for this timestep using deformed meshes (with rotation + translation)
    surfaces_t = {
        name: apply_transform(mesh_list[ct_idx])
        for name, mesh_list in deformed_meshes.items()
    }
    
    # Compute flow for each surface
    for name, mesh in surfaces_t.items():
        surf = mesh.extract_surface().clean()
        
        if surf.n_cells == 0:
            surf = pv.PolyData(surf.points).delaunay_2d()
        
        surf = surf.compute_cell_sizes(length=False, area=True, volume=False)
        
        try:
            surf = surf.compute_normals(cell_normals=True, point_normals=False, inplace=False)
        except TypeError:
            surf = pv.PolyData(surf.points).delaunay_2d()
            surf = surf.compute_cell_sizes(length=False, area=True, volume=False)
            surf = surf.compute_normals(cell_normals=True, point_normals=False, inplace=False)
        
        # Sample velocity at points for storage (used later for writing output file)
        sampled_points = surf.sample(vel_grid_t)
        deformed_flow_t[name].append(sampled_points["Velocity"])
        

deformed_flow = deformed_flow_t.copy()


if n_frames < 3:
    raise ValueError("at least three MRI frames are required for cycle shifting and closure")

for name, mesh in surfaces_t.items():
    for i in range(n_frames):
        if i==0:    #first
            deformed_flow[name][i]= deformed_flow_t[name][len(deformed_flow_t[name])-1]

        elif i == n_frames - 2:    # penultimate frame
            deformed_flow[name][i] = (deformed_flow_t[name][n_frames - 3] - deformed_flow[name][0]) / 2
            
        elif i==len(deformed_flow_t[name])-1:     #last
            deformed_flow[name][i]= deformed_flow[name][0]

        elif i < len(deformed_flow_t[name])-1:
            deformed_flow[name][i]= deformed_flow_t[name][i+1]

In [ ]:
# ============================================================
# Compute flow rates using stored deformed_flow velocities
# ============================================================

flow_rates = {
    "inlet": [],
    "out": [],
    "out1": [],
    "out2": [],
    "out3": [],
}

for t in tqdm(range(n_frames), desc="Computing flow rates"):

    # Match MRI timestep to CT timestep
    ct_idx = min(int(t * n_ct_meshes / n_frames), n_ct_meshes - 1)

    # Current timestep surfaces
    surfaces_t = {
        name: apply_transform(mesh_list[ct_idx])
        for name, mesh_list in deformed_meshes.items()
    }

    for name, mesh in surfaces_t.items():

        surf = mesh.extract_surface().clean()

        if surf.n_cells == 0:
            surf = pv.PolyData(surf.points).delaunay_2d()

        # Areas
        surf = surf.compute_cell_sizes(
            length=False,
            area=True,
            volume=False
        )

        # Normals
        try:
            surf = surf.compute_normals(
                cell_normals=True,
                point_normals=False,
                inplace=False
            )

        except TypeError:

            surf = pv.PolyData(surf.points).delaunay_2d()

            surf = surf.compute_cell_sizes(
                length=False,
                area=True,
                volume=False
            )

            surf = surf.compute_normals(
                cell_normals=True,
                point_normals=False,
                inplace=False
            )

        # ----------------------------------------------------
        # Point velocities already stored in deformed_flow
        # ----------------------------------------------------
        point_velocity = deformed_flow[name][t]

        # Convert point data -> cell data
        cell_velocity = np.zeros((surf.n_cells, 3))

        for c in range(surf.n_cells):

            ids = surf.get_cell(c).point_ids
            cell_velocity[c] = point_velocity[ids].mean(axis=0)

        # ----------------------------------------------------
        # Flow rate
        # ----------------------------------------------------
        normals = surf.cell_data["Normals"]
        areas = surf.cell_data["Area"]

        vn = np.sum(cell_velocity * normals, axis=1)

        Q = np.sum(vn * areas)
        flow_rates[name].append(Q)

In [ ]:

for i in flow_rates["inlet"]:
    print(i)

In [ ]:

min_distance = 0.5  # mm
# Find points closer than min_distance and mark them
inlet_transformed = apply_transform(inlet_mesh)
points = inlet_transformed.points

# Build KD-tree for efficient distance queries
tree = cKDTree(points)

# For each point, find points within min_distance
close_points_mask = np.zeros(len(points), dtype=bool)
for i, pt in enumerate(points):
    neighbors = tree.query_ball_point(pt, r=min_distance)
    # If any neighbor other than itself exists, mark as close
    if any(n != i for n in neighbors):
        close_points_mask[i] = True


n_close = np.sum(close_points_mask)
print(f"Found {n_close} points closer than {min_distance} mm to another point")

# Build a point cloud of the close points for visualization
if n_close > 0:
    close_points = pv.PolyData(points[close_points_mask])
else:
    close_points = pv.PolyData(np.empty((0, 3)))

# Visualization
p = pv.Plotter(notebook=True)
p.add_mesh(apply_transform(inlet_mesh), show_edges=True, color="cyan", opacity=0.7, label="inlet")
if n_close > 0:
    p.add_mesh(close_points, color="red", point_size=8, render_points_as_spheres=True, label="close points")
p.add_legend()
p.show()

In [ ]:
# Save inlet velocity vectors for each timestep as VTK files
output_dir = os.path.join(cfg["sim_dir"], "inlet_vtk")
os.makedirs(output_dir, exist_ok=True)

n_timesteps = len(deformed_flow["inlet"])
for t in range(n_timesteps):
    mesh_t = inlet_meshes_deformed[t].copy()
    mesh_t["Velocity"] = deformed_flow["inlet"][t]
    # Transform mesh to MRI coordinate system
    mesh_t_transformed = apply_transform(mesh_t)
    vel_grid_t = build_velocity_grid(velTemp, t, spacing_mm, origin_mm)
    
    # Sample velocity from MRI grid at transformed points
    sampled = mesh_t_transformed.sample(vel_grid_t)
    
    # Check if points are inside MRI bounds, zero out velocities outside
    bounds = vel_grid_t.bounds  # (xmin, xmax, ymin, ymax, zmin, zmax)
    pts = mesh_t_transformed.points
    
    inside_mask = (
        (pts[:, 0] >= bounds[0]) & (pts[:, 0] <= bounds[1]) &
        (pts[:, 1] >= bounds[2]) & (pts[:, 1] <= bounds[3]) &
        (pts[:, 2] >= bounds[4]) & (pts[:, 2] <= bounds[5])
    )
    
    # Zero out velocities for points outside MRI domain
    velocities = sampled["Velocity"].copy()
    velocities[~inside_mask] = [0.0, 0.0, 0.0]
    
    # Sampling occurs in the MRI frame, whereas GlobalNodeID belongs to the
    # original solver/CTA mesh. Rotate vector components back; translation does
    # not act on vectors.
    mesh_t["Velocity"] = rotate_vectors_to_source(velocities, R_Tra)
    mesh_t.save(os.path.join(output_dir, f"inlet_t{t:03d}.vtk"))

In [ ]:
print("inlet temporal and spacial inlet ")

# Write the inlet vectors through a tested helper. MRI velocities are in mm/s;
# the svFSI boundary file uses cm/s.
output_file = cfg["inlet_velocity_out"]
ref_mesh = inlet_meshes_deformed[0]
inlet_velocity_solver_frame = rotate_vectors_to_source(
    np.asarray(deformed_flow["inlet"]), R_Tra
)
time_steps = write_inlet_velocity_file(
    output_file,
    ref_mesh.point_data["GlobalNodeID"],
    inlet_velocity_solver_frame,
    close_points_mask,
    cfg["cycle_duration_s"],
)
print(cfg["cycle_duration_s"], len(time_steps), time_steps)


In [ ]:
flow_rates_ml_s = {
    name: np.array(fr) * 1e-3
    for name, fr in flow_rates.items()
}

In [ ]:
# Plot all surfaces (raw and smoothed)
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

colors = {'inlet': 'blue', 'out': 'red', 'out1': 'green', 'out2': 'orange', 'out3': 'purple'}

for ax, (name, fr_ml) in zip(axes, flow_rates_ml_s.items()):
    
    ax.plot(np.arange(fr_ml.size), fr_ml, marker="o", color=colors.get(name, 'black'), markersize=3, alpha=0.4, label='raw')
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel("Time step")
    ax.set_ylabel("Flow (mL/s)")
    ax.set_title(f"{name} flow")
    ax.grid(True)
    ax.legend(fontsize=8)
    if name == "inlet":
        ax.invert_yaxis()

# Combined plot in last subplot
ax_all = axes[-1]
for name, fr_smooth in flow_rates_ml_s.items():
    ax_all.plot(np.arange(len(fr_smooth)), fr_smooth, label=name, linewidth=2)
ax_all.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
ax_all.set_xlabel("Time step")
ax_all.set_ylabel("Flow (mL/s)")
ax_all.set_title("All surfaces (smoothed)")
ax_all.legend()
ax_all.grid(True)

plt.tight_layout()
plt.show()

# Check mass conservation using smoothed data
inlet_mean = np.mean(flow_rates_ml_s["inlet"])
outlet_mean=[]
for name in ["out", "out1", "out2", "out3"]:
    if name=="out":
        fr_ml=flow_rates_ml_s[name]
    else:
        fr_ml=flow_rates_ml_s[name]

        
    outlet_mean.append(np.mean(fr_ml))


outlets_mean=sum(outlet_mean)
print(f"\nMass conservation check (smoothed):")
print(f"  Inlet mean:   {inlet_mean:.3f} mL/s")
print(f"  out0 mean:  {outlet_mean[0]:.3f} mL/s")
print(f"  out1 mean:  {outlet_mean[1]:.3f} mL/s")
print(f"  out2 mean:  {outlet_mean[2]:.3f} mL/s")

print(f"  out3 mean:  {outlet_mean[3]:.3f} mL/s")
print(f"  Outlets sum:  {outlets_mean:.3f} mL/s")
print(f"  Difference:   {abs(inlet_mean) - abs(outlets_mean):.3f} mL/s ({100*abs(abs(inlet_mean) - abs(outlets_mean))/abs(inlet_mean):.1f}%)")



In [ ]:
print(f"\nMass conservation check (smoothed):")
print(f"  Inlet mean:   {inlet_mean:.3f} mL/s")
# Adjust outlet means to conserve mass
total_outlet = sum(outlet_mean)
scale_factor = abs(inlet_mean) / abs(total_outlet)
outlet_mean = [om * scale_factor for om in outlet_mean]

print(f"  out0 mean:  {outlet_mean[0]:.3f} mL/s. Expected: {-inlet_mean*0.7:.3f} mL/s")
print(f"  out1 mean:  {outlet_mean[1]:.3f} mL/s. Expected: {-inlet_mean*0.15:.3f} mL/s")
print(f"  out2 mean:  {outlet_mean[2]:.3f} mL/s. Expected: {-inlet_mean*0.09:.3f} mL/s")
print(f"  out3 mean:  {outlet_mean[3]:.3f} mL/s. Expected: {-inlet_mean*0.06:.3f} mL/s")
total_outlet = sum(outlet_mean)
print(f"  Difference:   {abs(inlet_mean) - abs(total_outlet):.3f} mL/s ({100*abs(abs(inlet_mean) - abs(total_outlet))/abs(inlet_mean):.1f}%)")

In [ ]:
# Print inlet flow (inverted so inflow is positive)
fr =flow_rates_ml_s["inlet"]  # Invert signal


cardiac_cycle = 60.0 / mean_hr  # ~0.857 s
n_frames = len(fr)
dt = cardiac_cycle / n_frames  # time step duration

# Create time array starting from dt (not 0)
time_array = np.array([(t + 1) * dt for t in range(n_frames)])
print(len(fr), "16")

print("Inlet flow (time [s], flow [mL/s]):")
for t_sec, fr_val in zip(time_array, fr):
    print(f" {t_sec:.6f} {fr_val:.6f}")
print("\n")


fig, ax = plt.subplots()
ax.plot(time_array, fr, marker="o", color=colors.get(name, 'black'), markersize=3, alpha=0.4, label='raw')
ax.invert_yaxis()
ax.grid(True)
plt.show()


print("\nFlow rate summary (mm³/s):")
for name in surfaces_t:
    print(f"  {name:6s}: mean={np.mean(flow_rates_ml_s[name]):10.2f}, max={np.max(flow_rates_ml_s[name]):10.2f}, min={np.min(flow_rates_ml_s[name]):10.2f}")
